# Transformer

## Attention

In [ ]:
import torch
import torch.nn
import torch.nn.functional as F
L=5 # Time (9am, 12pm, etc)
d_in=4 # These are the variables/features that we are measuring (Humidity, Temperature, etc)
d_head=2
n_heads=3
d_model=d_head*n_heads # 6
X = torch.rand(L,d_in)
WQ = torch.rand(d_model,d_in)
Q = X@WQ.T # (L,d_model)
WK = torch.rand(d_model,d_in)
K = X@WK.T # (L,d_model)
WV = torch.rand(d_model,d_in)
V = X@WV.T # (L,d_model)

sdpa_reference=F.scaled_dot_product_attention(Q,K,V)
scores = (Q@K.T)/(d_model**0.5)
weights = scores.softmax(axis=1)
sdpa_manual = weights@V
torch.allclose(sdpa_reference,sdpa_manual)

True

## Multi-Head-Attention

In [ ]:
mha_model=torch.nn.MultiheadAttention(d_model,num_heads=n_heads,bias=False,batch_first=True)
attention_reference,_=mha_model(Q,K,V)
mha_model.in_proj_weight.shape

#print(model.in_proj_weight.shape) # (3*d_model,d_model)
WQ_mha=mha_model.in_proj_weight[:d_model,:]
WQ_H1=WQ_mha[:d_head,:]
WQ_H2=WQ_mha[d_head:2*d_head,:]
WQ_H3=WQ_mha[2*d_head:,:]
WK_mha=mha_model.in_proj_weight[d_model:d_model*2,:]
WK_H1=WK_mha[:d_head,:]
WK_H2=WK_mha[d_head:2*d_head,:]
WK_H3=WK_mha[2*d_head:,:]
WV_mha=mha_model.in_proj_weight[d_model*2:,:]
WV_H1=WV_mha[:d_head,:]
WV_H2=WV_mha[d_head:2*d_head,:]
WV_H3=WV_mha[2*d_head:,:]

Q1=Q@WQ_H1.T
Q2=Q@WQ_H2.T
Q3=Q@WQ_H3.T

K1 = K @ WK_H1.T
K2 = K @ WK_H2.T
K3 = K @ WK_H3.T

V1 = V@WV_H1.T
V2 = V@WV_H2.T
V3 = V@WV_H3.T

#H1:
#head1 = (Q1@K1.T)/d_head
#weights1=head1.softmax(dim=1)
#output1=weights1@V1
head1=F.scaled_dot_product_attention(Q1,K1,V1)
head2=F.scaled_dot_product_attention(Q2,K2,V2)
head3=F.scaled_dot_product_attention(Q3,K3,V3)

h=torch.cat([head1,head2,head3],dim=1) # (L,d_model)

WO = mha_model.out_proj.weight.data
out_final = h@WO.T
torch.allclose(out_final,attention_reference)

True

## Transformer Encoder

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
torch.manual_seed(1)
d_in=4 # Input feature groups (temperature, humidity, windspeed, and rain percentage)
L = 5 # Readings in a week
n_heads=3 # Number of attention heads
d_head=2 # Features per head
d_model= n_heads*d_head # 6
d_feedforward = 8

X = torch.rand(L,d_in)
X_proj = X@torch.rand(d_in,d_model) # (L, d_model)

EncoderLayer = nn.TransformerEncoderLayer(d_model=d_model, nhead = n_heads, dim_feedforward=d_feedforward, dropout=0, batch_first=True, bias=True)
out_reference_encoder = EncoderLayer(X_proj)


In [ ]:
### Multihead Attention
my_self_attn = nn.MultiheadAttention(d_model,n_heads, batch_first=True)

my_self_attn.in_proj_weight.data = EncoderLayer.self_attn.in_proj_weight.data
my_self_attn.out_proj.weight.data = EncoderLayer.self_attn.out_proj.weight.data

my_self_attn.in_proj_bias.data = EncoderLayer.self_attn.in_proj_bias.data
my_self_attn.out_proj.bias.data = EncoderLayer.self_attn.out_proj.bias.data

out_attention,_ = my_self_attn(X_proj, X_proj, X_proj) # Since Q,K,V are the same as X_proj, we are preforming self-attention.

### Add & Norm
my_norm1 = nn.LayerNorm(d_model)
out_norm1 = my_norm1(X_proj+out_attention)

### Feed Forward Neural Network
my_linear1 = nn.Linear(in_features=d_model,out_features=d_feedforward, bias=True)
my_linear2 = nn.Linear(in_features=d_feedforward, out_features=d_model, bias=True)

my_linear1.weight.data = EncoderLayer.linear1.weight.data
my_linear2.weight.data = EncoderLayer.linear2.weight.data
my_linear1.bias.data = EncoderLayer.linear1.bias.data
my_linear2.bias.data = EncoderLayer.linear2.bias.data

out_linear1 = my_linear1(out_norm1)
out_relu = F.relu(out_linear1)
out_linear2 = my_linear2(out_relu)

### Add & Norm
my_norm2 = nn.LayerNorm(d_model)
out_final = my_norm2(out_norm1+out_linear2) # (L, d_model)

torch.allclose(out_reference_encoder, out_final)

True

## Transformer Decoder

In [ ]:
import torch
import torch.nn as nn
DecoderLayer = nn.TransformerDecoderLayer(d_model=d_model,nhead=n_heads, dim_feedforward=d_feedforward, dropout=0, batch_first=True, bias=True)
out_reference_decoder = DecoderLayer(X_proj, X_proj)

### Self Attention
my_self_attn = nn.MultiheadAttention(d_model,n_heads, batch_first=True)

my_self_attn.in_proj_weight.data = DecoderLayer.self_attn.in_proj_weight.data
my_self_attn.out_proj.weight.data = DecoderLayer.self_attn.out_proj.weight.data

my_self_attn.in_proj_bias.data = DecoderLayer.self_attn.in_proj_bias.data
my_self_attn.out_proj.bias.data = DecoderLayer.self_attn.out_proj.bias.data

out_self_attention,_ = my_self_attn(X_proj, X_proj, X_proj) # Since Q,K,V are the same as X_proj, we are preforming self-attention.

### Add & Norm 1
my_norm1 = nn.LayerNorm(d_model)
out_norm1 = my_norm1(X_proj+out_self_attention)

### Cross Attention
my_multihead_attn = nn.MultiheadAttention(d_model, n_heads, batch_first=True)

my_multihead_attn.in_proj_weight.data = DecoderLayer.multihead_attn.in_proj_weight.data
my_multihead_attn.out_proj.weight.data = DecoderLayer.multihead_attn.out_proj.weight.data

my_multihead_attn.in_proj_bias.data = DecoderLayer.multihead_attn.in_proj_bias.data
my_multihead_attn.out_proj.bias.data = DecoderLayer.multihead_attn.out_proj.bias.data

out_cross_attn,_ = my_multihead_attn(out_norm1, X_proj,X_proj)

### Add & Norm 2
my_norm2 = nn.LayerNorm(d_model)
out_norm2 = my_norm2(out_norm1+out_cross_attn)

### Feed Forward Neural Network
my_linear1 = nn.Linear(in_features=d_model,out_features=d_feedforward, bias=True)
my_linear2 = nn.Linear(in_features=d_feedforward, out_features=d_model, bias=True)

my_linear1.weight.data = DecoderLayer.linear1.weight.data
my_linear2.weight.data = DecoderLayer.linear2.weight.data
my_linear1.bias.data = DecoderLayer.linear1.bias.data
my_linear2.bias.data = DecoderLayer.linear2.bias.data

out_linear1 = my_linear1(out_norm2)
out_relu = F.relu(out_linear1)
out_linear2 = my_linear2(out_relu)

### Add & Norm 3
my_norm3 = nn.LayerNorm(d_model)
out_final = my_norm3(out_norm2+out_linear2)

torch.allclose(out_final, out_reference_decoder)



True